In [ ]:
import signac
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymser
from matplotlib.backends.backend_pdf import PdfPages

from utils.molec_class_files import esolvs
from Build_GPs.utils.signac import get_signac_results, save_signac_results
from Build_GPs.utils.id_new_samples import new_samples_vle, find_pareto, new_samples_ld, check_mse_10
from Build_GPs.utils.models import get_best_models
from Build_GPs.utils.plot import plot_gp_examples
import pickle

from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real, values_scaled_to_real_tf, values_real_to_scaled_tf

In [ ]:
import os
import tensorflow as tf

def sp_within_bounds(analyzer):
    """
    Check if the state point parameters are within the molecule's bounds
    """
    param_bounds, param_names = analyzer.get_param_bnds_names()
    #Get the max sigma from the bounds and names
    sigma_vals = [v for n, v in zip(param_names, param_bounds) if "sigma" in n]
    max_sigma = np.max(sigma_vals)
    param_bnds2 = analyzer.values_real_to_pref(param_bounds.T).T
    all_molec_dir = analyzer.use_dir_name
    if os.path.exists(os.path.join(all_molec_dir, "best_per_run.csv")):
        all_df = pd.read_csv(os.path.join(all_molec_dir, "best_per_run.csv"), header=0, index_col=0)
    #Get the best set where no bound is approached
    all_data = all_df
    first_param_name = param_names[0] + "_cum"
    last_param_name = param_names[-1] + "_cum"
    param_vals = all_data.loc[:, first_param_name:last_param_name].values
    # Find which bounds are different
    lower_bnd = param_bnds2[:, 0]
    upper_bnd = param_bnds2[:, 1]
    dif_bnds = lower_bnd != upper_bnd
    # Check closeness to bounds for params that have variable bounds
    close_to_lower = np.isclose(param_vals[:,dif_bnds], lower_bnd[dif_bnds])
    close_to_upper = np.isclose(param_vals[:,dif_bnds], upper_bnd[dif_bnds])
    close_any = np.logical_or(close_to_lower, close_to_upper)
    # A "valid" row has no True in close_any
    valid_rows = ~close_any.any(axis=1)
    # Pick first valid row or the first row if none are valid
    best_idx = np.argmax(valid_rows) if valid_rows.any() else 0
    return best_idx

from Build_GPs.utils.models import get_exp_data, loo_model_perform
mol_names = ["DEC", "DMF", "DMSO", "EG", "Gly", "MeOH"] 
molec_dict = esolvs.make_dict(mol_names)

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/Opt_ES")
from utilsOpt import opt_atom_types
#For each of the 6 solvents
data_dict = []
#Load GP models
max_mapd = 0
mode = "w_temp" #w_temp or wo_temp
for mol_name in mol_names:
    at_num = 0 
    setup = opt_atom_types.Problem_Setup([mol_name], at_num, "ExpVal")
    # Set parameter set of interest (in this case get the best parameter set)
    x_label = "best_set"
    all_molec_dir = setup.use_dir_name
    GPs = setup.all_gp_dict[mol_name]
    ranks = {}
    q_vals = {}
    for prop_name in ["liq_density", "surf_tens"]:
        GP_model = GPs[f"sim_{prop_name}"]
        mol_data = molec_dict[mol_name]
        exp_data, property_bounds, property_name = get_exp_data(mol_data, prop_name)

        #Get parameter sets and property values for best parameter set
        if prop_name == "liq_density":
            ms_data = pd.read_csv(f"../Build_GPs/analysis/{mol_name}/ld_iters/all_results.csv")
        else:            
            ms_data = pd.read_csv(f"../Build_GPs/analysis/{mol_name}/vle_iters/all_results.csv")

        param_bounds, param_names = setup.get_param_bnds_names()
        first_param_name = param_names[0]
        last_param_name = param_names[-1]
        # best_idx = sp_within_bounds(setup)

        real_best_set = ms_data.loc[:, first_param_name:last_param_name]
        real_temp = ms_data.loc[:, "temperature"]
        real_prop = ms_data.loc[:, f"{prop_name}"]
        X_test = np.hstack((real_best_set, real_temp.values.reshape(-1,1)))
        
        #Calculate the Sensitivity Matrix
        X_test_tf = tf.Variable(X_test, dtype=tf.float64)

        with tf.GradientTape() as tape:
            tape.watch(X_test_tf)
            x_data = X_test_tf
            mean, _ = GP_model.predict_f(x_data)            
        
        sensitivity_matrix = tape.gradient(mean, X_test_tf)
        gp_input_names = param_names + ["temperature"]

        #Remove last column of sensitivity matrix if we don't want to include temperature
        if mode == "wo_temp":
            sensitivity_matrix = sensitivity_matrix[:, :-1]
            names_to_use = param_names
        else:
            names_to_use = gp_input_names

        #SVD of sensitivity matrix to get singular values and right basis vectors
        S, U, Vt = tf.linalg.svd(sensitivity_matrix)
        sing_vals = S.numpy()
        right_vecs = Vt.numpy().T  # Transpose to get eigenvectors as columns
        q = sing_vals@abs(right_vecs)

        #Reorder input names based on the magntude of the sensitivity (q)
        sorted_indices = np.argsort(q)[::-1]
        #sorted values of q
        sorted_q = q[sorted_indices]

        
        param_ranks = [names_to_use[i] for i in sorted_indices]
        ranks[prop_name] = param_ranks
        q_vals[prop_name] = sorted_q

        #Save singular values, and right singular vectors as csvs to dir
        if prop_name == "liq_density":
            dir = f"../Build_GPs/analysis/{mol_name}/ld_iters/sens_approx"
        else:
            dir = f"../Build_GPs/analysis/{mol_name}/vle_iters/sens_approx"
        if mode == "wo_temp":
            name_dir = "_wo_temp"
        else:            
            name_dir = ""
        sens_dir_name = "sens_approx" + name_dir
        os.makedirs(os.path.join(dir, sens_dir_name), exist_ok=True) 
        sens_dir = os.path.join(dir, sens_dir_name)
        singval_df = pd.DataFrame(sing_vals, columns=["sing_value"])
        singval_df.to_csv(os.path.join(sens_dir, f"sing_val_{prop_name}.csv"), index=False)
        rightvec_df = pd.DataFrame(right_vecs, columns=[f"{gp_input_names[i]}" for i in range(right_vecs.shape[1])])
        rightvec_df.to_csv(os.path.join(sens_dir, f"basis_vec_{prop_name}.csv"), index=False)
        rank_df = pd.DataFrame({"param_rank": param_ranks,"sensitivity_score": sorted_q})        
        rank_df.to_csv(os.path.join(sens_dir, f"param_rank_{prop_name}.csv"), index=False)
        print(np.linalg.norm(right_vecs, axis=0))

    #Create a table which shows the parameter ranks for each property and save to csv
    dir = f"../Build_GPs/analysis/{mol_name}/"
    rank_table = pd.DataFrame({"Rank": list(range(1, len(param_ranks) + 1)), 
                               r"\rho_l - LD Data": ranks["liq_density"],
                               "q_j (rho_l)": q_vals["liq_density"],
                                r"\gamma - ST Data": ranks["surf_tens"],
                                "q_j (gamma)": q_vals["surf_tens"]})
    rank_table.to_csv(os.path.join(dir, f"param_rank_table{name_dir}.csv"), index=False)


In [ ]:
#Compare GP Prediction Accuracy to MAPD of final FF model
import os
from Build_GPs.utils.models import get_exp_data, loo_model_perform
mol_names = ["DEC", "DMF", "DMSO", "EG", "Gly", "MeOH"] 
molec_dict = esolvs.make_dict(mol_names)

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/Opt_ES")
from utilsOpt import opt_atom_types
#For each of the 6 solvents
data_dict = []
#Load GP models
max_mapd = 0
for mol_name in mol_names:
    at_num = 0 
    setup = opt_atom_types.Problem_Setup([mol_name], at_num, "ExpVal")
    # Set parameter set of interest (in this case get the best parameter set)
    x_label = "best_set"
    all_molec_dir = setup.use_dir_name
    GPs = setup.all_gp_dict[mol_name]
    for prop_name in ["liq_density", "surf_tens"]:
        GP_model = GPs[f"sim_{prop_name}"]
        mol_data = molec_dict[mol_name]

        #get the MAPD of final FF model predictions on test set
        #load from Opt_ES/analysis/at_00/MeOH/ExpVal/opt_res/ms_val_opt/error_data.csv
        error_data = pd.read_csv(f"{setup.use_dir_name}/ms_val_opt/error_data.csv")
        ff_mapd = error_data[f'mapd_{prop_name}'].max()
        exp_data, property_bounds, property_name = get_exp_data(mol_data, prop_name)

        #Or get x and y data from the org parameter sets and property values used to train the GP model
        #Get parameter sets and property values for best parameter set
        ms_data = pd.read_csv(f"{setup.use_dir_name}/ms_val_opt/ms_data.csv")
        param_bounds, param_names = setup.get_param_bnds_names()
        first_param_name = param_names[0]
        last_param_name = param_names[-1]
        real_best_set = ms_data.loc[:, first_param_name:last_param_name].values
        real_temp = ms_data.loc[:, "temperature"].values
        real_prop = ms_data.loc[:, f"sim_{prop_name}"].values
        #scale the parameter sets to be between 0 and 1 using the parameter bounds
        scaled_best_set = values_real_to_scaled(real_best_set, param_bounds)
        temperature_bounds = mol_data.temperature_bounds(f"expt_{prop_name}")
        scaled_temp = values_real_to_scaled(real_temp, temperature_bounds)
        scaled_prop = values_real_to_scaled(real_prop, property_bounds)
        x_data = np.hstack((scaled_best_set, scaled_temp))
        y_data = scaled_prop
        
        #Calculate MAPD of GP model predictions on the parameter sets and property values used to train the GP model
        __, GP_mapd = loo_model_perform({prop_name: GP_model}, x_data, y_data, property_bounds)
        
        data_dict.append([mol_name, prop_name, property_name, GP_mapd, ff_mapd])
        if np.maximum(GP_mapd, ff_mapd) > max_mapd:
            max_mapd = np.maximum(GP_mapd, ff_mapd)
    
#Plot the results on a scatter plot with MAPD of GP on x-axis and MAPD of FF on y-axis. 2 subplots for each property
data_df = pd.DataFrame(data_dict, columns=["Molecule", "Property", "Property Name", "GP_MAPD", "FF_MAPD"])
fig, axes = plt.subplots(1, 2, figsize=(12,6))
for i, property_name in enumerate(data_df["Property Name"].unique()):
    label_type = "o" if i == 0 else "s"
    prop_data = data_df[data_df["Property Name"] == property_name]
    colors = plt.cm.tab10.colors
    for j in range(len(prop_data)):
        axes[i].scatter(prop_data["GP_MAPD"].iloc[j], prop_data["FF_MAPD"].iloc[j], 
                        label=prop_data["Molecule"].iloc[j], 
                        color=colors[j % len(colors)], marker=label_type, s=150, alpha = 0.5)
    axes[i].plot(prop_data["GP_MAPD"], prop_data["GP_MAPD"], color="black", label = "Parity Line")
    axes[i].set_title(f"{property_name.split('/')[0]} MAPD Comparison", fontsize = 24)
    if i == 1:
        axes[i].set_ylim(0, max_mapd*1.05)
    else:
        axes[i].legend(loc="upper center", fontsize = 18)
    axes[i].tick_params("y", direction="inout", which="both", length=7)
    axes[i].tick_params("y", which="major", length=14)
    axes[i].tick_params("x", pad=15)
    #Increase font size of ticks    
    axes[i].tick_params(axis='both', which='major', labelsize=14)
fig.supylabel('Actual FF MAPD/%', fontsize=18)
fig.supxlabel('GP-Predicted FF MAPD/%', fontsize=18)

plt.tight_layout()

#Save the figure
save_path = os.path.join("analysis", "AT-0", "ms_val_opt", 'MAPD_comp_bestFF.png')
print(f"Saving figure to {save_path}")
fig.savefig(save_path)

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import re
import os

### Functions to set tick positions and labels for mpl

def set_ticks_for_axis(ax, param_bounds, nticks):
    """Set the tick positions and labels on y axis for each plot

    Tick positions based on normalised data
    Tick labels are based on original data
    """
    min_val, max_val = param_bounds
    step = (max_val - min_val) / float(nticks-1)
    tick_labels = [round(min_val + step * i, 2) for i in range(nticks)]
    ticks = np.linspace(0, 1.0, nticks)
    ax.yaxis.set_ticks(ticks)
    ax.set_yticklabels(tick_labels, fontsize=16)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.tick_params("y", direction="inout", which="both", length=7)
    ax.tick_params("y", which="major", length=14)
    ax.tick_params("x", pad=15) 

def set_ticks_for_axis(ax, param_bounds, nticks):
    import numpy as np
    from matplotlib import ticker

    min_val, max_val = param_bounds

    # Base ticks
    ticks = np.linspace(0.0, 1.0, nticks)
    values = np.linspace(min_val, max_val, nticks)

    # --- FORCE zero tick if it lies in bounds ---
    if min_val < 0 < max_val:
        zero_tick = (0.0 - min_val) / (max_val - min_val)

        ticks = np.append(ticks, zero_tick)
        values = np.append(values, 0.0)

        # Sort consistently
        order = np.argsort(ticks)
        ticks = ticks[order]
        values = values[order]

    # Apply ticks
    ax.yaxis.set_ticks(ticks)
    y_labels_rounded = []
    for v in values:
        if abs(v) >= 1.0:
            y_labels_rounded.append(round(v, 1))
        else:
            y_labels_rounded.append(round(v, 2))
    ax.set_yticklabels(y_labels_rounded, fontsize=16)

    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.tick_params("y", direction="inout", which="both", length=7)
    ax.tick_params("y", which="major", length=14)
    ax.tick_params("x", pad=15)


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import re
import os

### Functions to calculate parameter sums and make correlation matrices for all molecules together

def calc_param_sums(df_analyze, data_class, mol_names, mode3 = "scl"):
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B

    param_names = list(data_class.param_names)
    project = signac.get_project("Build_GPs/vle_iters")
    #Grab system.top from any the first vle iter results
    jobs = project.find_jobs({"mol_name": mol_names[0], "iter": 1})
    job = None
    for i, job_first in enumerate(jobs):
        job = job_first
        break
    #Open system.top and read the atom types
    # Extract unique atom types from param names
    atom_types = set(p.split("_")[1] for p in param_names)

    # Initialize counts dictionary
    param_counts = {atype: 0 for atype in atom_types}

    # Load .top file
    top_file = job.fn("system.top")
    with open(top_file, "r") as f:
        lines = f.readlines()

    # Step 1: extract atoms section to count atom types
    # GROMACS .top atoms section starts with [ atoms ] and ends at next [ ... ]
    atoms_section = []
    inside_atoms = False
    for line in lines:
        line = line.strip()
        if line.startswith("[ atoms ]"):
            inside_atoms = True
            continue
        if inside_atoms:
            if line.startswith("[") or line == "":
                break
            atoms_section.append(line)

    # Parse atoms lines (assume standard GROMACS format: nr type resnr resid atom cgnr charge mass)
    for line in atoms_section:
        if line.startswith(";") or line == "":
            continue
        parts = re.split(r"\s+", line)
        atom_type = parts[1]  # second column is atom type
        if atom_type in param_counts:
            param_counts[atom_type] += 1
    #For each row, get the sum of the rows of sigma 
    sigma_cols = [c for c in df_analyze.columns if c.startswith("sigma")]
    epsilon_cols = [c for c in df_analyze.columns if c.startswith("epsilon")]
    #Scale sigma and epsilon between 0 and 1
    #Get data for sigma and epsilon in array form
    if mode3 == "scl":
        data = df_analyze[sigma_cols + epsilon_cols].values
        #Scale data
        data = values_real_to_scaled(data, data_class.param_bounds)
        data[data > 1e5] = 0
        df_analyze[sigma_cols + epsilon_cols] = data
    elif mode3 == "from_scl":
        #Make values zero whenever bounds are less than 1e-8 apart
        data = df_analyze[sigma_cols + epsilon_cols].values
        param_bnds = np.array(data_class.param_bounds)  # shape (n_params, 2)
        lower_bnd = param_bnds[:, 0]
        upper_bnd = param_bnds[:, 1]
        fixed_cols = np.isclose(upper_bnd, lower_bnd, rtol=1e-8)
        data[:, fixed_cols] = 0
        df_analyze[sigma_cols + epsilon_cols] = data
    if mode3 == "to_real":
        data = df_analyze[sigma_cols + epsilon_cols].values
        #Scale data
        data = values_scaled_to_real(data, data_class.param_bounds)
        data[data < 1e-5] = 0
        df_analyze[sigma_cols + epsilon_cols] = data

    #Calculate weighted sums
    df_analyze["sigma_sum"] = sum(df_analyze[col] * param_counts[col.split("_")[1]] for col in sigma_cols) #* NM_TO_ANGSTROM
    df_analyze["epsilon_sum"] = sum(df_analyze[col] * param_counts[col.split("_")[1]] for col in epsilon_cols) #* KJMOL_TO_K
    if mode3 == "real" or mode3 == "to_real":
        df_analyze["sigma_sum"] = df_analyze["sigma_sum"]* NM_TO_ANGSTROM
        df_analyze["epsilon_sum"] = df_analyze["epsilon_sum"]* KJMOL_TO_K
    return df_analyze, param_counts, param_names

def weighted_min_max(param_bounds_slice, param_names_slice, param_counts):
    x_min = np.sum([
        param_bounds_slice[i, 0] * param_counts[param_names_slice[i].split("_")[1]]
        for i in range(len(param_names_slice))
    ])
    x_max = np.sum([
        param_bounds_slice[i, 1] * param_counts[param_names_slice[i].split("_")[1]]
        for i in range(len(param_names_slice))
    ])
    return x_min, x_max

def get_corr_all_molecs(mol_names, mode, mode2 = "all", mode3 ="scl", err_met = "mpd", threshold=10):
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B
    import matplotlib.pyplot as plt 

    all_df_analysis = pd.DataFrame()
    x_min_sig = np.inf
    x_max_sig = -np.inf
    x_min_eps = np.inf
    x_max_eps = -np.inf
    for molec in mol_names:
        # ID the top ten by lowest average MAPE
        molec_dict = esolvs.make_dict(mol_names)
        data_class = molec_dict[molec]
        if mode == "ld":
            if mode2 == "pareto":
                df = pd.read_csv("Build_GPs/analysis/" + molec + "/ld_iters/mse-less10-full.csv", header = 0, index_col=0)
            else:
                df_all_res = pd.read_csv("Build_GPs/analysis/"+molec+"/ld_iters/all_results.csv", header = 0, index_col=0)
                #For each group of param names and temperature, calculate the average mpd_liq_density
                #Remove all rows where 5 temperatures do not have ld results
                df_all_res = df_all_res.dropna().copy()
                if molec == "MeOH":
                    df_all_res = df_all_res[df_all_res["iter"] <= 1]
                df_all_res = df_all_res.groupby(list(data_class.param_names)).filter(lambda x: len(x) >= 5) 
                #Calculate the average mpd_liq_density for each group
                df_all_res["expt_liq_density"] = df_all_res["temperature"].apply(
                lambda x: data_class.expt_liq_density[x])
                df_all_res["pct_err"] = ((df_all_res["liq_density"] - df_all_res["expt_liq_density"]) / df_all_res["expt_liq_density"]) * 100
                df = (df_all_res.groupby(list(data_class.param_names)).agg(mpd=("pct_err", "mean")).reset_index())
            
        elif mode == "vle":
            props_pareto = ["liq_density", "surf_tens"] 
            if mode2 == "pareto":
                df_pareto = pd.read_csv("Build_GPs/analysis/" + molec + "/vle_iters/iter-1/pareto-params.csv", header = 0, index_col=0)
                #Get only the lowest mapd value row where pareto is true
                df_final = df_pareto.drop(columns="is_pareto")
            else:
                df_final = pd.read_csv("Build_GPs/analysis/" + molec + "/vle_iters/iter-1/result_errors.csv", header = 0, index_col=0)
            props_mse = ["mapd_" + prop for prop in props_pareto]
            df= df_final.copy()
            # df = df_final[df_final[props_mse].le(threshold).all(axis=1)

        df_analyze = df.copy()
        #For each row, get the sum of the rows of sigma 
        df_analyze, param_counts, param_names = calc_param_sums(df_analyze, data_class, mol_names, mode3 = mode3)

        if molec == "DEC" and mode == "vle" and mode2 == "all":
            df_analyze = df_analyze[df_analyze["sigma_sum"] > 5]

        #Drop columns not containing sum or mpd
        df_analyze = df_analyze.loc[:, df_analyze.columns.str.contains("sum|mpd")]
        df_analyze["Molecule"] = molec
        if mode == "ld":
            df_analyze.rename(columns={"mpd": "mpd_liq_density"}, inplace=True)
        
        all_df_analysis = pd.concat([all_df_analysis, df_analyze], ignore_index=True)
        
        data = df[list(data_class.param_names)].values
        data = values_real_to_scaled(data, data_class.param_bounds)
        param_bounds = data_class.param_bounds
        indx_mid = int(len(data_class.param_names) / 2)
        if mode3 == "scl":
            param_bounds = values_real_to_scaled(param_bounds.T, data_class.param_bounds).T #For consistency
        if mode3 != "scl":
            param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
            param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K
        # Split param_names to match sigma / epsilon
        sigma_names = param_names[:indx_mid]
        epsilon_names = param_names[indx_mid:]

        # Compute weighted sums
        x_min_sig_new, x_max_sig_new = weighted_min_max(param_bounds[:indx_mid], sigma_names, param_counts)
        x_min_eps_new, x_max_eps_new = weighted_min_max(param_bounds[indx_mid:], epsilon_names, param_counts)

        if x_min_sig_new < x_min_sig:
            x_min_sig = x_min_sig_new
        if x_max_sig_new > x_max_sig:
            x_max_sig = x_max_sig_new
        if x_min_eps_new < x_min_eps:
            x_min_eps = x_min_eps_new
        if x_max_eps_new > x_max_eps:
            x_max_eps = x_max_eps_new

    if mode == "ld":
        use_df = all_df_analysis.drop(columns="Molecule") #df_new
    else:
        use_df = all_df_analysis.drop(columns=["Molecule", "mpd_liq_density"])
    print(use_df)
    meth_corr = "spearman"

    corr_matrix = use_df.corr(method=meth_corr)
    
    return all_df_analysis, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps


def plot_corr_all_molecs(all_df_analysis, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mode2 = "all"):
    # Get default color cycle
    cmap = plt.get_cmap("tab10")
    fig2, axes = plt.subplots(1, 3, figsize=(15,5))
    ax1, ax2, ax3 = axes.flat[:3]
    for i, (values, group) in enumerate(all_df_analysis.groupby("Molecule")):
        c = cmap(i)
        if mode == "ld":
            ax1.plot(group["sigma_sum"], group["mpd_liq_density"], 'o', color = c, label=values, alpha=0.7)
            ax2.plot(group["epsilon_sum"], group["mpd_liq_density"], 'o', color = c, alpha=0.7)
        elif mode == "vle":
            ax1.plot(group["sigma_sum"], group["mpd_surf_tens"], 'o', color = c, alpha=0.7)
            ax2.plot(group["epsilon_sum"], group["mpd_surf_tens"], 'o', color = c, label=values, alpha=0.7) 
        ax3.plot(group["sigma_sum"], group["epsilon_sum"], 'o', color = c, alpha=0.7)

    if mode == "ld":  
        ax1.set_xlabel(r"$\Sigma \sigma$/A", fontsize=32)
        ax1.set_ylabel(r"MPD $\rho_l$/%", fontsize=32)
        ax1.tick_params(axis='both', which='major', labelsize=18)
        if mode == "vle":
            ax1.set_ylim(-5, 5)
        ax1.set_xlim(x_min_sig, x_max_sig)

        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=32)
        ax2.set_ylabel(r"MPD $\rho_l$/%", fontsize=32)
        # if mode == "vle":
        #     ax4.set_ylim(-5, 5)
        ax2.tick_params(axis='both', which='major', labelsize=18)
        ax2.set_xlim(x_min_eps, x_max_eps)

        

    elif mode == "vle":
        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=32)
        ax2.set_ylabel(r"MPD $\gamma$/%", fontsize=32)
        ax2.tick_params(axis='both', which='major', labelsize=18)
        ax2.set_xlim(x_min_eps, x_max_eps)

        ax1.set_xlabel(r"$\Sigma \sigma$/A", fontsize=32)
        ax1.set_ylabel(r"MPD $\gamma$/%", fontsize=32)
        ax1.tick_params(axis='both', which='major', labelsize=18)
        ax1.set_xlim(x_min_sig, x_max_sig)

    ax3.set_xlabel(r"$\Sigma \sigma$/A", fontsize=32)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=32)
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=18)
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig2.tight_layout()
    fig2.legend(loc = 'upper center', fontsize=20, ncol=6, bbox_to_anchor=(0.5, 1.15)  )
    return fig2
    
    
def plot_corr_matrix(corr_matrix):
    import seaborn as sns
    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(8,8))
    size = 20

    rename_map = {}
    for i, col in enumerate(corr_matrix.columns):
        # Map the current column name to the corresponding letter
        if "sigma_sum" in col:
            rename_map[col] = r"$\Sigma \sigma$/A"
        elif "epsilon_sum" in col:
            rename_map[col] = r"$\Sigma \frac{\epsilon}{k_B}$/K"
        elif "mpd_liq_density" in col:
            rename_map[col] = "MPD " + r"$\rho_l$" + "/%"
        elif "mpd_surf_tens" in col:
            rename_map[col] = "MPD " + r"$\gamma$" + "/%"
        else:
            rename_map[col] = col
    corr_matrix = corr_matrix.rename(columns=rename_map, index=rename_map)
    ax = sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=size)    
    ax.set_xlabel(ax.get_xlabel(), fontsize=32)
    ax.set_ylabel(ax.get_ylabel(), fontsize=32)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    plt.title("Correlation Matrix", fontsize=size)
    # plt.show()
    return fig

In [ ]:
### Make correlation matrices for all molecules
err_met = "mpd" # or mapd
mode2 = "all"  # or "pareto"
mode3 = "scl"
threshold = 10
mol_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/")
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")
#Make pdf 
full_at_dir = os.path.join(f"Build_GPs/analysis/{'-'.join(mol_names)}")
os.makedirs(full_at_dir, exist_ok=True)
modes = ["ld", "vle"]
for mode in modes:
    all_df_analysis, corr_matrix, x_min_sig, x_max_sig, x_min_eps, x_max_eps = get_corr_all_molecs(mol_names, mode, mode2, mode3, err_met, threshold)
    pdf_hpvap = PdfPages(os.path.join(full_at_dir ,f"{mode}_corr_{mode2}.pdf"))
    print(os.path.join(full_at_dir ,f"{mode}_corr_{mode2}.pdf"))
    pdf_hpvap.savefig(plot_corr_all_molecs(all_df_analysis, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mode2), bbox_inches='tight')
    plt.close()
    pdf_hpvap.savefig(plot_corr_matrix(corr_matrix), bbox_inches='tight')
    plot_corr_matrix(corr_matrix)
    plt.close()
    pdf_hpvap.close()

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import signac
from Opt_ES.utilsOpt import opt_atom_types
import re
import glob

### Functions to make correlation matrices for each individual molecule

def get_corr_one_molec(mol_name, mode, mode2 = "all", mode3 ="scl", err_met = "mpd", threshold=10):
    mol_names = [mol_name]
    molec_dict = esolvs.make_dict(mol_names)
    data_class = molec_dict[mol_name]
    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B

    x_min_sig = np.inf
    x_max_sig = -np.inf
    x_min_eps = np.inf
    x_max_eps = -np.inf

    # ID the top ten by lowest average MAPE
    if mode == "ld":
        if mode2 == "pareto":
            df = pd.read_csv("Build_GPs/analysis/" + mol_name + "/ld_iters/mse-less10-full.csv", header = 0, index_col=0)
        else:
            df_all_res = pd.read_csv("Build_GPs/analysis/"+mol_name+"/ld_iters/all_results.csv", header = 0, index_col=0)
            #For each group of param names and temperature, calculate the average mpd_liq_density
            #Remove all rows where 5 temperatures do not have ld results
            df_all_res = df_all_res.dropna().copy()
            #Remove all rows where iter > 2
            if mol_name == "MeOH":
                df_all_res = df_all_res[df_all_res["iter"] <= 1]
            df_all_res = df_all_res.groupby(list(data_class.param_names)).filter(lambda x: len(x) >= 5) 
            #Calculate the average mpd_liq_density for each group
            df_all_res["expt_liq_density"] = df_all_res["temperature"].apply(
            lambda x: data_class.expt_liq_density[x])
            df_all_res["pct_err"] = ((df_all_res["liq_density"] - df_all_res["expt_liq_density"]) / df_all_res["expt_liq_density"]) * 100
            df = (df_all_res.groupby(list(data_class.param_names)).agg(mpd=("pct_err", "mean")).reset_index())
    elif mode == "vle":
        props_pareto = ["liq_density", "surf_tens"] 
        if mode2 == "pareto":
            df_pareto = pd.read_csv("Build_GPs/analysis/" + mol_name + "/vle_iters/iter-1/pareto-params.csv", header = 0, index_col=0)
            #Get only the lowest mapd value row where pareto is true
            df_final = df_pareto.drop(columns="is_pareto")
        else:
            df_final = pd.read_csv("Build_GPs/analysis/" + mol_name + "/vle_iters/iter-1/result_errors.csv", header = 0, index_col=0)
        props_mse = ["mapd_" + prop for prop in props_pareto]
        df= df_final.copy()
        # df = df_final[df_final[props_mse].le(threshold).all(axis=1)

    df_analyze = df.copy()
    #For each row, get the sum of the rows of sigma 
    df_analyze, param_counts, param_names = calc_param_sums(df_analyze, data_class, mol_names, mode3 = mode3)
    if mode == "ld":
        df_analyze.rename(columns={"mpd": "mpd_liq_density"}, inplace=True)

    if mol_name == "DEC" and mode == "vle" and mode2 == "all":
        df_analyze = df_analyze[df_analyze["sigma_sum"] > 5]

    #Drop columns not containing sum or mpd
    df_new = df_analyze.loc[:, df_analyze.columns.str.contains("sigma|epsilon|mpd")].copy()
    df_analyze = df_analyze.loc[:, df_analyze.columns.str.contains("sum|mpd")]

    seaborn.set_palette('bright', n_colors=len(df))
    data = df[list(data_class.param_names)].values
    data = values_real_to_scaled(data, data_class.param_bounds)
    data_scl = data.copy()

    if mode == "ld":
        max_error = df[f"{err_met}"].max()
        min_error = df[f"{err_met}"].min()
        result_bounds = np.array([[min_error, np.maximum(20, max_error)]])
        results = values_real_to_scaled(df[[f"{err_met}"]].values, result_bounds)
    elif mode == "vle":
        if err_met == "mapd":
            result_bounds = np.array([[0, 20], [0,20]])
        elif err_met == "mpd":
            err_cols = [c for c in df.columns if err_met in c]
            ld_error = err_cols[0]
            st_error = err_cols[1]
            max_error_ld = df[ld_error].max()
            min_error_ld = df[ld_error].min()
            max_error_st = df[st_error].max()
            min_error_st = df[st_error].min()
            result_bounds = np.array([[min_error_ld, max_error_ld], [min_error_st, max_error_st]])
        results = values_real_to_scaled(df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values, result_bounds)
    param_bounds = data_class.param_bounds
    indx_mid = int(len(data_class.param_names) / 2)
    if mode3 == "scl":
        param_bounds = values_real_to_scaled(param_bounds.T, data_class.param_bounds).T #For consistency
    if mode3 != "scl":
        param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
        param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K

    def weighted_min_max(param_bounds_slice, param_names_slice, param_counts):
        x_min = np.sum([
            param_bounds_slice[i, 0] * param_counts[param_names_slice[i].split("_")[1]]
            for i in range(len(param_names_slice))
        ])
        x_max = np.sum([
            param_bounds_slice[i, 1] * param_counts[param_names_slice[i].split("_")[1]]
            for i in range(len(param_names_slice))
        ])
        return x_min, x_max

    # Split param_names to match sigma / epsilon
    sigma_names = param_names[:indx_mid]
    epsilon_names = param_names[indx_mid:]

    # Compute weighted sums
    x_min_sig, x_max_sig = weighted_min_max(param_bounds[:indx_mid], sigma_names, param_counts)
    x_min_eps, x_max_eps = weighted_min_max(param_bounds[indx_mid:], epsilon_names, param_counts)

    data = np.hstack((data, results))
    bounds = np.vstack((param_bounds, result_bounds))

    col_names = []
    for name in data_class.param_names:
        latex_name = lambda s: fr"$\{s.split('_',1)[0]}_{{{s.split('_',1)[1]}}}$" if '_' in s else fr"${s}$"
        col_names.append(latex_name(name))

    err_met_upper = err_met.upper()
    if mode == "ld":
        col_names += [err_met_upper + "\n" + r"$\rho_l$"] #r"$\rho_l_{\mathrm{sat}}$"
    elif mode == "vle":
        col_names += [err_met_upper + "\n" + r"$\rho_l$", err_met_upper + "\n" + r"$\gamma$"]
    # print("Column names: ", col_names)
    n_axis = len(col_names)
    assert data.shape[1] == n_axis
    x_vals = [i for i in range(n_axis)]

    return df_analyze, data, bounds, data_class, col_names, n_axis, x_vals, x_min_sig, x_max_sig, x_min_eps, x_max_eps

def plot_corr_one_molec(data, bounds, data_class, col_names, n_axis, x_vals, mode, mol_name):
    import matplotlib.pyplot as plt 
    from matplotlib import ticker

    mol_names = [mol_name]
    molec_dict = esolvs.make_dict(mol_names)
    data_class = molec_dict[mol_name]
    # Create (N-1) subplots along x axis
    fig, axes = plt.subplots(1, n_axis-1, sharey=False, figsize=(20,5))

    # Plot each row
    for i, ax in enumerate(axes):
        for j, line in enumerate(data):
            ax.plot(x_vals, line, alpha=0.45)

        ax.set_xlim([x_vals[i], x_vals[i+1]])


    for dim, ax in enumerate(axes):
        ax.xaxis.set_major_locator(ticker.FixedLocator([dim]))
        set_ticks_for_axis(ax, bounds[dim], nticks=6)
        if dim < 10:
            ax.set_xticklabels([col_names[dim]], fontsize=24)
        else:
            ax.set_xticklabels([col_names[dim]], fontsize=20)
        ax.set_ylim(-0.05,1.05)
        # Add white background behind labels
        for label in ax.get_yticklabels():
            label.set_bbox(
                dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.45,
                    boxstyle=mpatch.BoxStyle("round4")
                )
            )
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_linewidth(2.0)

    ax = axes[-1]
    ax.xaxis.set_major_locator(ticker.FixedLocator([n_axis-2, n_axis-1]))
    ax.set_xticklabels([col_names[-2], col_names[-1]], fontsize=20)

    ax = plt.twinx(axes[-1])
    #Include zero as a tick on the last 2 y axes
    ticks = ax.get_yticks()          # or get_xticks()
    ticks = np.unique(np.append(ticks, 0))
    ax.set_ylim(-0.05, 1.05)
    set_ticks_for_axis(ax, bounds[-1], nticks=6)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['right'].set_linewidth(2.0)

    # Remove space between subplots
    plt.subplots_adjust(wspace=0, bottom=0.3)
    # ax.legend(fontsize=16)
    # plt.tight_layout()
    #Set title
    if mode == "ld":
        title_end = "LD Iters"
    elif mode == "vle":
        title_end = "ST Iter"
    plt.suptitle(f"LJ Parameter Sets for {mol_name} {title_end}", fontsize=20)
    return fig

def plot_corr_matrix_one_molec(df_analyze, data_class, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mol_name):
    import matplotlib.pyplot as plt
    fig2, axes = plt.subplots(1, 3, figsize=(15,5))
    ax1, ax2, ax3 = axes.flat[:3]
    if mode == "ld":
        for i in range(len(df_analyze)):
            ax1.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["mpd_liq_density"].iloc[i], 'o', label="Liquid Density")
        ax1.set_xlabel(r"$\Sigma \sigma$/A", fontsize=16)
        ax1.set_ylabel(r"MPD $\rho_l$/%", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        ax1.set_xlim(x_min_sig, x_max_sig)

        for i in range(len(df_analyze)):
            ax2.plot(df_analyze["epsilon_sum"].iloc[i], df_analyze["mpd_liq_density"].iloc[i], 'o', label="Liquid Density")
        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=16)
        ax2.set_ylabel(r"MPD $\rho_l$/%", fontsize=16)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        ax2.set_xlim(x_min_eps, x_max_eps)

    if mode == "vle":
        for i in range(len(df_analyze)):
            ax2.plot(df_analyze["epsilon_sum"].iloc[i], df_analyze["mpd_surf_tens"].iloc[i], 'o', label="Surface Tension")
        ax2.set_xlabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=16)
        ax2.set_ylabel(r"MPD $\gamma$/%", fontsize=16)
        ax2.tick_params(axis='both', which='major', labelsize=14)
        ax2.set_xlim(x_min_eps, x_max_eps)
        for i in range(len(df_analyze)):
            ax1.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["mpd_surf_tens"].iloc[i], 'o', label="Surface Tension")
        ax1.set_xlabel(r"$\Sigma \sigma$/A", fontsize=16)
        ax1.set_ylabel(r"MPD $\gamma$/%", fontsize=16)
        ax1.tick_params(axis='both', which='major', labelsize=14)
        ax1.set_xlim(x_min_sig, x_max_sig)


    for i in range(len(df_analyze)):
        ax3.plot(df_analyze["sigma_sum"].iloc[i], df_analyze["epsilon_sum"].iloc[i], 'o', label="Liquid Density")
    ax3.set_xlabel(r"$\Sigma \sigma$/A", fontsize=16)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=16)
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig2.tight_layout()

    from sklearn.preprocessing import StandardScaler, MinMaxScaler

    if mode == "ld":
        use_df = df_analyze #df_new
    else:
        use_df = df_analyze.drop(columns=["mpd_liq_density"])
    meth = "spearman"
    corr_matrix = use_df.corr(method=meth)  # or "spearman"

    import seaborn as sns
    import matplotlib.pyplot as plt
    size = 48/(len(corr_matrix))
    fig4 = plt.figure(figsize=(8,8))
    rename_map = {}
    for i, col in enumerate(corr_matrix.columns):
        # Map the current column name to the corresponding letter
        if "sigma_sum" in col:
            rename_map[col] = r"$\Sigma \sigma$/A"
        elif "epsilon_sum" in col:
            rename_map[col] = r"$\Sigma \frac{\epsilon}{k_B}$/K"
        elif "mpd_liq_density" in col:
            rename_map[col] = "MPD " + r"$\rho_l$" + "/%"
        elif "mpd_surf_tens" in col:
            rename_map[col] = "MPD " + r"$\gamma$" + "/%"
        else:
            rename_map[col] = col
    corr_matrix = corr_matrix.rename(columns=rename_map, index=rename_map)
    ax = sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=size) 
    ax.set_xlabel(ax.get_xlabel(), fontsize=size)
    ax.set_ylabel(ax.get_ylabel(), fontsize=size)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    
    plt.title("Correlation Matrix", fontsize=size)

    #Concatenate 
    wildcard = "1" if mol_name != "MeOH" else "1"
    lhs_param_df = pd.read_csv("Build_GPs/analysis/" + mol_name + "/" +mode +"_iters/params-iter-1.csv", header = 0, index_col=0) #
    csv_files = glob.glob(f"Build_GPs/analysis/{mol_name}/{mode}_iters/params-iter-{wildcard}.csv")
    # Read and concatenate all files
    lhs_param_df = pd.concat(
        [pd.read_csv(f, header=0, index_col=0) for f in csv_files],
        ignore_index=True
    ).drop_duplicates()

    print(len(lhs_param_df), " LHS points before removing duplicates with df_analyze")
    mode3_corr = "from_scl" if mode3 == "scl" else "to_real"
    mol_names = mol_name.split("-")
    lhs_param_df, param_counts, param_names = calc_param_sums(lhs_param_df, data_class, mol_names, mode3 = mode3_corr)

    lhs_array = lhs_param_df[["sigma_sum", "epsilon_sum"]].to_numpy()
    analyze_array = df_analyze[["sigma_sum", "epsilon_sum"]].to_numpy()

    # Set tolerance
    tol = 1e-8  # adjust as needed

    # Create a boolean mask for each lhs row: True if it's far from all df_analyze rows
    mask = np.array([
        not np.any(np.all(np.isclose(row, analyze_array, rtol=tol, atol=tol), axis=1))
        for row in lhs_array
    ])

    # Keep only the "far" rows
    failure_points = lhs_param_df[mask].reset_index(drop=True)
    failure_points = failure_points.loc[:, lhs_param_df.columns.str.contains("sum")]
    lhs_param_df = lhs_param_df.loc[:, lhs_param_df.columns.str.contains("sum")]
    corr_mat_df = lhs_param_df
    print(len(lhs_param_df), " LHS points remain after removing duplicates with df_analyze")
    print(len(df_analyze), " points in df_analyze")
    corr_matrix2 = corr_mat_df.corr(method=meth)  # or "spearman"
    fig5 = plt.figure(figsize=(8,8))
    size = np.maximum(10,48/(len(corr_matrix2)))

    for i, col in enumerate(corr_matrix2.columns):
        # Map the current column name to the corresponding letter
        if "sigma_sum" in col:
            rename_map[col] = r"$\Sigma \sigma$/A"
        elif "epsilon_sum" in col:
            rename_map[col] = r"$\Sigma \frac{\epsilon}{k_B}$/K"
        elif "mpd_liq_density" in col:
            rename_map[col] = "MPD " + r"$\rho_l$" + "/%"
        elif "mpd_surf_tens" in col:
            rename_map[col] = "MPD " + r"$\gamma$" + "/%"
        else:
            rename_map[col] = col
    corr_matrix2 = corr_matrix2.rename(columns=rename_map, index=rename_map)
    ax = sns.heatmap(corr_matrix2, annot=True, cmap="coolwarm", center=0, fmt=".2f", annot_kws={"size": size})
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=size) 
    ax.set_xlabel(ax.get_xlabel(), fontsize=size)
    ax.set_ylabel(ax.get_ylabel(), fontsize=size)
    ax.tick_params(axis='x', labelsize=size)
    ax.tick_params(axis='y', labelsize=size)
    plt.title("Correlation Matrix (LHS Samples)", fontsize=size)
    plt.show()

    fig3, ax3 = plt.subplots(1, 1, figsize=(5,5))
    ax3.plot(df_analyze["sigma_sum"], df_analyze["epsilon_sum"], 'o', label="Successful Points", alpha = 0.5)
    ax3.plot(failure_points["sigma_sum"], failure_points["epsilon_sum"], 'o', label="Failed Points", alpha = 0.5)
    ax3.set_xlabel(r"$\Sigma \sigma$/A", fontsize=16)
    ax3.set_ylabel(r"$\Sigma \frac{\epsilon}{k_B}$/K", fontsize=16)
    ax3.set_ylim(x_min_eps, x_max_eps)
    ax3.tick_params(axis='both', which='major', labelsize=14)
    ax3.set_xlim(x_min_sig, x_max_sig)
    fig3.legend(loc = 'upper center', fontsize=16, ncol=2, bbox_to_anchor=(0.5, 1.15)  )
    fig3.tight_layout()
    if mode == "ld":
        return fig2, fig3, fig4, fig5
    elif mode == "vle":
        return fig2, fig3, fig4

In [ ]:
###Plot correlation matrices for each individual molecule

err_met = "mpd" # or mapd
mode2 = "all"  # or "pareto"
mode3 = "scl"
threshold = 10
mol_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/")
matplotlib.rc("font", family="sans-serif")
matplotlib.rc("font", serif="Arial")
#Make pdf 
modes = ["ld", "vle"]
for mode in modes:
# for mode in modes:
    for mode2 in ["all", "pareto"]:
        for molec_name in mol_names:   
            full_at_dir = os.path.join(f"Build_GPs/analysis/{molec_name}/{mode}_iters")
            os.makedirs(full_at_dir, exist_ok=True)
            pdf_hpvap = PdfPages(os.path.join(full_at_dir ,f"{mode}_corr_{mode2}.pdf"))
            df_analyze, data, bounds, data_class, col_names, n_axis, x_vals, x_min_sig, x_max_sig, x_min_eps, x_max_eps = get_corr_one_molec(molec_name, mode, mode2, "real", err_met, threshold) 
            fig_par = plot_corr_one_molec(data, bounds, data_class, col_names, n_axis, x_vals, mode, molec_name)
            pdf_hpvap.savefig(fig_par, bbox_inches='tight')
            plt.close()
            df_analyze, data, bounds, data_class, col_names, n_axis, x_vals, x_min_sig, x_max_sig, x_min_eps, x_max_eps = get_corr_one_molec(molec_name, mode, mode2, mode3, err_met, threshold) 
            figs = plot_corr_matrix_one_molec(df_analyze, data_class, x_min_sig, x_max_sig, x_min_eps, x_max_eps, mode, mol_name = molec_name) 
            for fig in figs:
                pdf_hpvap.savefig(fig, bbox_inches='tight')
                plt.close()   
            pdf_hpvap.close()

In [ ]:
#Compare parameters and error metrics for each molecule on parallel plot for ST iterations

import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
import seaborn
from matplotlib import ticker
import os
from utils.molec_class_files import esolvs
from fffit.fffit.utils import values_real_to_scaled, values_scaled_to_real
os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/Opt_ES")
from utilsOpt import opt_atom_types


molec_names = ["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"] #Change me as needed

dir_name = f"analysis/AT-0/ms_val_opt/"
os.makedirs(dir_name, exist_ok=True)
pdf_name = os.path.join(dir_name , f"parallel_plot.pdf")
pdf = PdfPages(pdf_name)

for str in molec_names:
    at_num = 0
    err_met = "mpd"
    mol_names = str.split("-") # ["EG", "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC", "DMF"] Change me as needed

    molec_dict = esolvs.make_dict(mol_names)
    matplotlib.rc("font", family="sans-serif")
    matplotlib.rc("font", serif="Arial")

    NM_TO_ANGSTROM = 10
    K_B = 0.008314 # J/MOL K
    KJMOL_TO_K = 1.0 / K_B

    # ID the top ten by lowest average MAPE
    #Get params < 10
    # df = pd.read_csv("analysis/at_01/EG-Gly-MeOH/ExpVal/opt_res/best_per_run.csv", header = 0, index_col=0)
    def get_ms_best(visual, mol, param_bnds, param_names, all_molec_dir, err_met):
        path_best_sets = os.path.join(all_molec_dir, "best_per_run.csv")
        df = pd.read_csv(path_best_sets, header=0)
        first_param_name = param_names[0] + "_cum"
        last_param_name = param_names[-1] + "_cum"
        data = df.loc[0, first_param_name:last_param_name].values
        data = visual.values_pref_to_real(data.reshape(1,-1))
        data = values_real_to_scaled(data.reshape(1,-1), param_bnds)

        #Load the data for the param set found by NW Method (from ms_val/pareto_params.csv)
        pareto_path = f"../Build_GPs/analysis/{mol}/vle_iters/iter-1/result_errors.csv"
        pareto_df = pd.read_csv(pareto_path, header=0)
        min_obj_idx = abs(pareto_df[f"{err_met}_surf_tens"]).idxmin()
        min_obj_idx2 = (
        pareto_df[f"{err_met}_surf_tens"]
        .abs()
        .sort_values()
        .index[1]
    )
        pareto_params = pareto_df.loc[:, param_names[0]:param_names[-1]].values
        data_pareto = values_real_to_scaled(pareto_params, param_bnds)

        #Get the results from validation files
        path_to_opt_ms_val = os.path.join(all_molec_dir, "ms_val_opt/error_data.csv")
        df_val_res = pd.read_csv(path_to_opt_ms_val, header=0)
        results = df_val_res[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values

        err_cols = [c for c in pareto_df.columns if err_met in c]
        result_bounds = np.zeros((len(err_cols),2))
        for col in err_cols:
            col_max = max(np.abs(pareto_df[col].values).max(), np.abs(df_val_res[col].values).max())
            if err_met == "mpd":
                col_min = -col_max
            else:
                col_min = 0
            result_bounds[err_cols.index(col)] = [col_min, col_max]        

        pareto_vals = pareto_df[[f"{err_met}_liq_density", f"{err_met}_surf_tens"]].values
        results_pareto = values_real_to_scaled(pareto_vals, result_bounds)
        results = values_real_to_scaled(results.reshape(1,-1), result_bounds).reshape(1,-1)
        # results = np.full(shape=len(results_pareto.T), fill_value=np.nan).reshape(1,-1)
        return data, data_pareto, results, results_pareto, result_bounds, min_obj_idx, min_obj_idx2

    visual = opt_atom_types.Vis_Results(mol_names, at_num, 1, "ExpVal")
    param_bnds, param_names = visual.get_param_bnds_names()
    # Set parameter set of interest (in this case get the best parameter set)
    x_label = "best_set"
    all_molec_dir = visual.use_dir_name
    data_best, data_pareto, results_best, results_pareto, result_bounds, min_obj_idx, min_obj_idx2= get_ms_best(visual, str, param_bnds, param_names, all_molec_dir, err_met="mpd")
    #Print results shapes
    data= np.vstack((data_best, data_pareto))
    results = np.vstack((results_best, results_pareto))

    param_bounds = param_bnds
    indx_mid = int(len(param_names) / 2)
    param_bounds[:indx_mid] = param_bounds[:indx_mid] * NM_TO_ANGSTROM
    param_bounds[indx_mid:] = param_bounds[indx_mid:] * KJMOL_TO_K

    data = np.hstack((data, results))
    bounds = np.vstack((param_bounds, result_bounds))

    col_names = []
    for name in param_names:
        latex_name = lambda s: fr"$\{s.split('_',1)[0]}_{{{s.split('_',1)[1]}}}$" if '_' in s else fr"${s}$"
        col_names.append(latex_name(name))

    err_met_upper = err_met.upper()
    col_names += [err_met_upper + "\n" + r"$\rho_l$", err_met_upper + "\n" + r"$\gamma$"]
    n_axis = len(col_names)
    assert data.shape[1] == n_axis
    x_vals = [i for i in range(n_axis)]

    # Create (N-1) subplots along x axis
    fig, axes = plt.subplots(1, n_axis-1, sharey=False, figsize=(20,5))

    # Plot each row
    for i, ax in enumerate(axes):
        for j, line in enumerate(data):
            # print(x_vals, line)
            if j ==0:
                if i ==len(axes)-1:
                    ax.plot(x_vals, line, alpha=1, label="GP-Opt", color="purple", linewidth=3.0, zorder=500)
                else:
                    ax.plot(x_vals, line, alpha=1, color="purple", linewidth=3.0, zorder=500)
                    
            elif j ==min_obj_idx+1:
                if i ==len(axes)-1:
                    ax.plot(x_vals, line, alpha=1, label="Base", color="red", linestyle = "--", linewidth=2.5, zorder=501)
                else:
                    ax.plot(x_vals, line, alpha=1, color="red", linestyle = "--", linewidth=2.5, zorder=501)
                    
            elif j ==min_obj_idx2+1:
                if i ==len(axes)-1:
                    ax.plot(x_vals, line, alpha=1, label="ST FF with Lowest MAPD " + r"$\rho_l$", color="dodgerblue", linestyle = "--", linewidth=2.5, zorder=501)
                else:
                    ax.plot(x_vals, line, alpha=1, color="dodgerblue", linestyle = "--", linewidth=2.5, zorder=501)

            elif j == len(data)-1 and i == len(axes)-1:
                ax.plot(x_vals, line, alpha=0.25, label="Other ST FFs", color="gray")
            #For the last line, add labels
            else:
                ax.plot(x_vals, line, alpha=0.25)
        ax.set_xlim([x_vals[i], x_vals[i+1]])

    for dim, ax in enumerate(axes):
        ax.xaxis.set_major_locator(ticker.FixedLocator([dim]))
        set_ticks_for_axis(ax, bounds[dim], nticks=6)
        if dim < 10:
            ax.set_xticklabels([col_names[dim]], fontsize=24)
        else:
            ax.set_xticklabels([col_names[dim]], fontsize=20)
        ax.set_ylim(-0.05,1.05)
        # Add white background behind labels
        for label in ax.get_yticklabels():
            label.set_bbox(
                dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.45,
                    boxstyle=mpatch.BoxStyle("round4")
                )
            )
        ax.spines['top'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_linewidth(2.0)

    ax = axes[-1]
    #Put legend at top center of plot
    ax.xaxis.set_major_locator(ticker.FixedLocator([n_axis-2, n_axis-1]))
    ax.set_xticklabels([col_names[-2], col_names[-1]], fontsize=20)

    ax = plt.twinx(axes[-1])
    ax.set_ylim(-0.05, 1.05)
    set_ticks_for_axis(ax, bounds[-1], nticks=6)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['right'].set_linewidth(2.0)
    
    # Remove space between subplots
    #Add Titile
    plt.subplots_adjust(wspace=0, bottom=0.3)

    # 2. Calculate the "Visual Center" of the combined axes
    # This finds the left edge of the first axis and the right edge of the last axis
    all_axes_bbox = fig.get_tightbbox(fig.canvas.get_renderer())
    # Convert that bbox to figure coordinates
    fig_bbox = fig.transFigure.inverted().transform_bbox(all_axes_bbox)
    # The true horizontal center of your DATA area
    data_center = 100*(fig_bbox.x0 + fig_bbox.x1) / 2

    fig.suptitle(f"{str} - Comparison of GP-Optimized FF and Unoptimized ST Iter FFs", fontsize=24, y=1.10)
    leg = fig.legend(loc='upper center', bbox_to_anchor=(data_center, 1.03), ncol=4, fontsize=16, bbox_transform=fig.transFigure)
    pdf.savefig(fig, bbox_inches='tight')   # save one figure at a time

pdf.close()



In [ ]:
###Get ST iteration pareto info for each molecule

import signac
import sys
import os
from pathlib import Path
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

# Now import using package structure relative to ES-FFO root
from utils.molec_class_files import esolvs
from Build_GPs.utils.signac import get_signac_results, save_signac_results
from Build_GPs.utils.id_new_samples import new_samples_vle, find_pareto
from Build_GPs.utils.models import get_best_models
from Build_GPs.utils.plot import plot_gp_examples, plot_sim_exp
from utils.prep_ms_data import prepare_df_errors, prepare_df_props

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/Build_GPs")

#Set iters to analyze and properties to analyze
iters = [1]  # Change me as needed
property_names = ["liq_density", "surf_tens"]  # Change me as needed
all_names = ["EG", "Gly", "MeOH", "DMSO", "DEC", "DMF"]

for mol in all_names:
    mol_names = [mol]
    #Set seeds and preferences
    cl_shuffle_seed = 1  # classifier
    gp_shuffle_seed = 42  # GP seed 30 for Gly (36)
    dist_seed = 1  # Distance seed
    mapd_le = 10
    save_csv = False
    save_fig = False
    verbose = True
    show_num = 5
    mode = f"best_{show_num}" #"pareto" #"sing", "pareto", or "all"
    mode = "pareto"



    ##############################################################################
    ##############################################################################
    if mode == "pareto" or mode =="all":
        def get_best_set_data(molec_name, mode="pareto"):
            # Check the analysis folder for analysis/MolName/vle_iters folders
            # Find the highest params-iter-X.csv file
            pareto_sets = pd.read_csv(f"analysis/{molec_name}/vle_iters/iter-1/pareto-params.csv", header = 0, index_col = 0)
            all_data = pd.read_csv(f"analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
            #Get the row where the mapd_surf_tens column is lowest
            #Return the array of all parameters (ignore mapd columns)
            param_set = pareto_sets.drop(columns=[col for col in pareto_sets.columns if any(x in col for x in ["mapd", "mse", "mae"])])
            common_cols = list(set(param_set.columns) & set(all_data.columns))
            # Filter all_data to rows that match any row in param_set on common_cols
            pareto_data = all_data.merge(param_set[common_cols].drop_duplicates(), on=common_cols)
            new_data = pd.DataFrame(pareto_data)

            #Uncomment last line to get all ST sets
            if mode == "all":
                new_data = all_data

            
            return new_data
    elif mode == "sing":
        def get_best_set_data(molec_name, mode = "sing"):
            # Check the analysis folder for analysis/MolName/vle_iters folders
            # Find the highest params-iter-X.csv file
            pareto_sets = pd.read_csv(f"analysis/{molec_name}/vle_iters/iter-1/final-params.csv", header = 0, index_col = 0)
            all_data = pd.read_csv(f"analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
            #Get the row where the mapd_surf_tens column is lowest
            best_row = pareto_sets.loc[pareto_sets['mapd_surf_tens'].idxmin()]
            #Return the array of all parameters (ignore mapd columns)
            param_set = best_row.drop(labels=[col for col in best_row.index if "mapd" in col])
            #Find the final parameters with the lowest surface tension
            param_set = pd.DataFrame(param_set).T
            mask = (all_data[param_set.columns] == param_set.iloc[0]).all(axis=1)

            #Apply mask
            all_data = all_data[mask]
            all_data = all_data.sort_values(by='temperature', ascending=True)
            all_data = pd.DataFrame(all_data)
            return all_data
    else:
        def get_best_set_data(molec_name, mode = mode):
            # Check the analysis folder for analysis/MolName/vle_iters folders
            # Find the highest params-iter-X.csv file
            molec_dict = esolvs.make_dict(mol_names)
            molecule = molec_dict[molec_name] 
            # pareto_sets = pd.read_csv(f"analysis/{molec_name}/ld_iters/mse-less10-full.csv", header = 0, index_col = 0)
            #Map exp data to density
            all_data = pd.read_csv(f"analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
            ld_threshold = (min(list(molecule.expt_liq_density.values()))+ max(list(molecule.expt_vap_density.values()))) / 2
            df_all, df_liquid, df_vapor = prepare_df_props(all_data, molecule, ld_threshold, scale=False)
                        #Calculate MAPD between expt_liq_density and liq_density for each param set, 
            # Prepare error data to find pareto points
            pareto_sets = prepare_df_errors(df_liquid, molec_dict, molec_name)
            prop_sort = "mapd_" + props
            if "worst" in mode:
                top_sets = pareto_sets.sort_values(by=prop_sort).tail(show_num)
            else:
                top_sets = pareto_sets.sort_values(by=prop_sort).head(show_num)
            
            print("Top sets: ", top_sets)
            # Drop non-parameter columns
            exclude = ["mse", "mpd", "mapd", "mae"]

            param_cols = [c for c in top_sets.columns if not any(substr in c for substr in exclude)]
            top_sets_params = top_sets[param_cols]
            # print(top_sets_params)

            # Build a mask: True if a row in all_data matches ANY of the top 15 rows
            mask = np.zeros(len(all_data), dtype=bool)
            for _, row in top_sets_params.iterrows():
                mask |= (all_data[param_cols] == row).all(axis=1)

            # Filter
            all_data = all_data[mask].sort_values(by='temperature', ascending=True)
            all_data = pd.DataFrame(all_data)
            # print(all_data.columns)
            return all_data

    def make_pareto_plots(molec_name, mode="pareto", err_met="mapd"):
        # Check the analysis folder for analysis/MolName/vle_iters folders
        # Find the highest params-iter-X.csv file
        pareto_sets = pd.read_csv(f"analysis/{molec_name}/vle_iters/iter-1/pareto-params.csv", header = 0, index_col = 0)


        pareto_sets_errs = pareto_sets[[col for col in pareto_sets.columns if any(x in col for x in ["mapd", "mse", "mae", "mpd"])]]
        
        #Plot err_met_liq_density vs err_met_surf_tens for each row in pareto_sets
        fig, ax = plt.subplots(figsize=(6,6))
        for i in range(len(pareto_sets_errs)):
            ax.scatter(pareto_sets_errs.iloc[i][f"{err_met}_liq_density"], pareto_sets_errs.iloc[i][f"{err_met}_surf_tens"], s=100, label="Pareto Points")
        ax.tick_params("both", direction="in", which="both", length=4, labelsize=20, pad=10)
        ax.tick_params("both", which="major", length=8)
        ax.xaxis.set_ticks_position("both")
        ax.yaxis.set_ticks_position("both")
        ax.set_xlabel(rf"{err_met.upper()} $\rho_l$/%", fontsize=32)
        ax.set_ylabel(rf"{err_met.upper()} $\gamma$/%", fontsize=32)
        # ax.set_title(f"Pareto Front for {molec_name}", fontsize=32)
        for axis in ['top','bottom','left','right']:
            ax.spines[axis].set_linewidth(2.0)
        # ax.legend()
        # plt.show()
        return fig


    #Get Project
    iter_type = "vle_iters" 
    project = signac.get_project(iter_type)
    #Load class properies for each molecule in the FF
    molec_dict = esolvs.make_dict(mol_names)

    # Save DataFrame of all molecule data for each iteration
    df_all_molec = get_signac_results(project, molec_dict, property_names)
    df_all_molec = save_signac_results(df_all_molec, iter_type, save_csv)

    #Check pareto efficient samples for each molecule to see if there is one with < mapd_le (10)% error in all properties
    all_final_params = find_pareto(df_all_molec, molec_dict, property_names, mapd_le)

    for key, value in all_final_params.items():
        #If there are, we have the final parameters
        if len(value) > 0:
            print(f"{key}: Final parameters:")
            # print(value)
            param_names = list(molec_dict[key].param_names)
            #Make a fxn in utils.plot to plot predictions vs exp data for LD and ST
            dir_name = f"analysis/{key}/{iter_type}"
            os.makedirs(dir_name, exist_ok=True)
            pdf_name = os.path.join(dir_name , f"prop_preds_{mode}.pdf")
            pdf = PdfPages(pdf_name)

            fig2 = make_pareto_plots(key, mode="pareto", err_met="mapd")
            pdf.savefig(fig2, bbox_inches='tight')
            
            for props in ["liq_density", "surf_tens"]:
                best_data = get_best_set_data(key, mode) 
                #Calculate MAE for each property and plot
                best_data = get_best_set_data(key, mode)

                          
                fig = plot_sim_exp(molec_dict[key], best_data, props)
                pdf.savefig(fig, bbox_inches='tight')   # save one figure at a time
            #Make plots of MAE for rho vs gamma  


            pdf.close()

In [ ]:
### Show GP vs actual predictions for ST iters for each molecule
import os
import pandas as pd
from fffit.fffit.plot import plot_model_vs_exp
from utils.molec_class_files import esolvs
from matplotlib.backends.backend_pdf import PdfPages
from Build_GPs.utils.plot import plot_gp_examples, plot_sim_exp
from Opt_ES.utilsOpt import opt_atom_types

import matplotlib.pyplot as plt
import numpy as np

molec_list = ["EG", "MeOH", "Gly", "DMSO", "DEC", "DMF"] #["EG", "MeOH", "Gly", "DMSO", "DMF", "DEC"]
at_number = 0
seed = 1
obj_choice = "ExpVal"
molec_dict = esolvs.make_dict(molec_list) 

os.chdir("/groups/ed/group_members/Montana_Carlozo/ES-FFO/Opt_ES")
def get_best_set_data(molec_name, mode = "sing"):
    # Check the analysis folder for analysis/MolName/vle_iters folders
    # Find the highest params-iter-X.csv file
    pareto_sets = pd.read_csv(f"../Build_GPs/analysis/{molec_name}/vle_iters/iter-1/final-params.csv", header = 0, index_col = 0)
    all_data = pd.read_csv(f"../Build_GPs/analysis/{molec_name}/vle_iters/all_results.csv", header = 0, index_col = 0)
    #Get the row where the mapd_surf_tens column is lowest
    best_row = pareto_sets.loc[pareto_sets['mapd_surf_tens'].idxmin()]
    #Return the array of all parameters (ignore mapd columns)
    param_set = best_row.drop(labels=[col for col in best_row.index if "mapd" in col])
    #Find the final parameters with the lowest surface tension
    param_set = pd.DataFrame(param_set).T
    mask = (all_data[param_set.columns] == param_set.iloc[0]).all(axis=1)

    #Apply mask
    all_data = all_data[mask]
    all_data = all_data.sort_values(by='temperature', ascending=True)
    all_data = pd.DataFrame(all_data)
    return all_data


# Loop over molecules
for mode in ["sing", "all"]:
    for molec_name in molec_list:
        pdf = PdfPages(f"analysis/at_00/{molec_name}/ExpVal/opt_res/prop_pred/gp_vs_sim_vs_exp_{mode}.pdf")
        visual = opt_atom_types.Vis_Results(molec_name.split("-") , at_number, seed, obj_choice)
        param_bnds, param_names = visual.get_param_bnds_names()
        mol_data = molec_dict[molec_name]
        best_ift_data = get_best_set_data(molec_name)
        all_ms_data = pd.read_csv(f"../Build_GPs/analysis/{molec_name}/vle_iters/all_results.csv", header=0, index_col=0)
        best_ms_data = pd.read_csv(f"analysis/at_00/{molec_name}/ExpVal/opt_res/ms_val/ms_data.csv", header=0, index_col=0)
        
        if len(best_ms_data) > 0:
            #drop all rows after the first
            best_ms_data = best_ms_data.iloc[:15]

        ##Rename any columns in best_ms_data to remove the string "sim_" if the column has it
        best_ms_data = best_ms_data.rename(columns={c: c.replace("sim_", "") for c in best_ms_data.columns if "sim_" in c})
        all_ms_data = all_ms_data.rename(columns={c: c.replace("sim_", "") for c in all_ms_data.columns if "sim_" in c})

        first_param_name = param_names[0]
        last_param_name = param_names[-1]

        group_cols = ["temperature"] + param_names
        best_ms_data = (
            best_ms_data.groupby(group_cols)
            .agg(
                liq_density=("liq_density", "mean"),
                liq_density_unc=("liq_density", "std"),
                surf_tens=("surf_tens", "mean"),
                surf_tens_unc=("surf_tens", "std"),
            )
            .reset_index())
        # print("Best Data ST Unc mean:", molec_name, np.mean(best_ms_data["surf_tens_unc"]))
        all_ms_data = (
            all_ms_data.groupby(group_cols)
            .agg(
                liq_density=("liq_density", "mean"),
                liq_density_unc=("liq_density", "std"),
                surf_tens=("surf_tens", "mean"),
                surf_tens_unc=("surf_tens", "std"),
            )
            .reset_index())

        # Get GPs associated with each molecule
        molec_gps_dict = visual.all_train_gp_dict[molec_name]

        # for i in range(len(best_ms_data)):
        ms_test_set = best_ms_data.loc[0, first_param_name:last_param_name].values
        # print("MS Test Set:", ms_test_set)
        test_params = visual.get_best_results(molec_name, theta_guess=ms_test_set.reshape(1,-1))
        #remove the GAFF key if it exists
        if "GAFF" in test_params:
            del test_params["GAFF"]

        # Loop over gps (1 per property)
        for key in list(molec_gps_dict.keys()):
            key_nosim = key.replace("sim_", "")
            if mode == "all":
                data_labels = [ "GP-Opt (Sim)", "Base (Sim)", "All ST Sets (Sim)",]
                data = [best_ms_data, best_ift_data, all_ms_data]
            elif mode == "sing":
                data_labels = [ "GP-Opt (Sim)", "Base (Sim)",]
                data = [best_ms_data, best_ift_data]
            #Prepare other data to plot
            other_data = {}
            for d_label, df in zip(data_labels, data):
                other_data[d_label] = df[["temperature", f"{key_nosim}", f"{key_nosim}_unc"]]

            # other_data = {f"IFT (Sim)": best_ift_data[["temperature", f"{key_nosim}", f"{key_nosim}_unc"]], f"Optimized (Sim)": best_ms_data[["temperature", f"{key_nosim}", f"{key_nosim}_unc"]]}
            # Set label
            # Get GP associated with property
            gp_model = molec_gps_dict[key]
            # Get X and Y data and bounds associated with the GP
            exp_data, y_bounds, y_names = visual.get_exp_data(mol_data, key)
            x_data = np.array(list(exp_data.keys()))
            y_data = np.array(list(exp_data.values()))

            # Plot test vs train for each parameter set
            pdf.savefig(plot_model_vs_exp(
                {molec_name: gp_model},
                test_params,
                exp_data,
                mol_data.temperature_bounds(),
                y_bounds,
                plot_bounds=mol_data.temperature_bounds(),
                property_name=y_names,
                other_data = other_data
                ), bbox_inches='tight')   # save one figure at a time
        pdf.close()